# 02 — OpenAI Function Calling API — Practical Examples

**Covers**: Full OpenAI API, strict mode, streaming FC, token budgeting, multi-tool pipeline

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Ready')

## 1. Full OpenAI FC Pipeline with Multiple Tools

In [ ]:
TOOLS = [
  {'type':'function','function':{'name':'get_stock_price','strict':True,'description':'Get current stock price for a ticker. Use for stock/finance questions.','parameters':{'type':'object','properties':{'ticker':{'type':'string','description':'Stock ticker e.g. AAPL, TSLA'}},'required':['ticker'],'additionalProperties':False}}},
  {'type':'function','function':{'name':'calculate','strict':True,'description':'Evaluate a math expression. Use for any arithmetic calculation.','parameters':{'type':'object','properties':{'expression':{'type':'string','description':'Python math expression e.g. 185.50 * 0.15'}},'required':['expression'],'additionalProperties':False}}}
]

def get_stock_price(ticker: str) -> str:
    prices = {'AAPL': 185.50, 'TSLA': 248.00, 'MSFT': 420.00, 'GOOGL': 175.20, 'NVDA': 875.00}
    p = prices.get(ticker.upper())
    return f'${p:.2f}' if p else f'Ticker {ticker} not found'

def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'Math error: {e}'

TOOL_MAP = {'get_stock_price': get_stock_price, 'calculate': calculate}

def run_agent(user_input: str) -> str:
    messages = [
        {'role': 'system', 'content': 'You are a financial assistant. Use tools for real data.'},
        {'role': 'user', 'content': user_input}
    ]
    for step in range(6):
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, tool_choice='auto', temperature=0)
        msg = r.choices[0].message
        messages.append(msg)
        if r.choices[0].finish_reason == 'tool_calls':
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = TOOL_MAP[tc.function.name](**args)
                print(f'  Step {step+1}: {tc.function.name}({args}) → {result}')
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        else:
            return msg.content
    return 'max steps'

print(run_agent('What is Apple stock price? If I own 100 shares, what is my portfolio value and what would a 15% gain be worth?'))

## 2. Checking finish_reason

In [ ]:
test_inputs = [
    'What is 2 + 2?',
    'What is Apple stock price?',
]
for inp in test_inputs:
    r = client.chat.completions.create(model=MODEL, messages=[{'role': 'user', 'content': inp}], tools=TOOLS, max_tokens=50)
    fr = r.choices[0].finish_reason
    has_tool = bool(r.choices[0].message.tool_calls)
    print(f'Input: "{inp}" → finish_reason: {fr}, tool_called: {has_tool}')

## 3. Token Usage Tracking

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'What is MSFT stock price?'}],
    tools=TOOLS
)
usage = r.usage
print(f'Prompt tokens:     {usage.prompt_tokens:>6}')
print(f'Completion tokens: {usage.completion_tokens:>6}')
print(f'Total tokens:      {usage.total_tokens:>6}')
cost = (usage.prompt_tokens * 0.00015 + usage.completion_tokens * 0.0006) / 1000
print(f'Cost (USD):        ${cost:.6f}')
print(f'At 1000 runs/day → ${cost*1000:.3f}/day')

## 4. Streaming Function Calls

In [ ]:
stream = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Get Apple stock price'}],
    tools=TOOLS, stream=True
)
buffer = {}
for chunk in stream:
    delta = chunk.choices[0].delta if chunk.choices else None
    if delta and delta.tool_calls:
        for tc_d in delta.tool_calls:
            i = tc_d.index
            if i not in buffer:
                buffer[i] = {'id': '', 'name': '', 'arguments': ''}
            if tc_d.id: buffer[i]['id'] += tc_d.id
            if tc_d.function.name: buffer[i]['name'] += tc_d.function.name
            if tc_d.function.arguments: buffer[i]['arguments'] += tc_d.function.arguments
            print(f'\rStreaming args: {buffer[i]["arguments"]}', end='')
print()
for idx, tc in buffer.items():
    print(f'\nFinal call: {tc["name"]}({json.loads(tc["arguments"])})')